In [2]:
import os
import sys
# load env
from dotenv import load_dotenv
load_dotenv()
# load env
from data_scraper import StockDataScraper
import pandas as pd
import plotly.express as px
import pandas as pd

In [ ]:

import requests
url=f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey={os.environ['ALPHA_VANTAGE_API_KEY']}&outputsize=full&datatype=json"
response = requests.get(url)
data = response.json()
data

In [2]:
from data_scraper import StockDataScraper
scraper=StockDataScraper(stockSymbol='MSFT')
scraper.getStockData()
scraper.dumpIntoCSV()

2025-03-31 20:28:07 - StockDataScraper - INFO - Fetching stock data for MSFT...
2025-03-31 20:28:10 - StockDataScraper - INFO - Fetched stock data for MSFT successfully...
2025-03-31 20:28:10 - StockDataScraper - INFO - Data dumped into /home/steeldev/Desktop/Github/PotatoTrade/ML/stock_data/MSFT_data.csv successfully...


In [3]:
df=scraper.structured_stock_data

In [2]:
import requests
import os
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
from data_scraper import StockDataScraper
from custom_logger import CustomLogger
import datetime
from custom_exceptions import raise_custom_exception, ScraperError, CustomException

class StockDataAnalyser:
    """
    Class for analysing stock data
    """
    def __init__(self, stock_symbol):
        self.stock_symbol = stock_symbol
        timestamp = datetime.datetime.now().strftime("%Y%m%d")
        self.logger = CustomLogger("StockDataAnalyser", log_file=f'./Logs/stock_data_analyser/{timestamp}.log').get_logger()
        self.stock_data=None

    def fetch_and_plot_stock_data(self):
        try:
            
            self.logger.info(f"Fetching stock data for {self.stock_symbol}...")
            
            try:
                scraper = StockDataScraper(stockSymbol=self.stock_symbol)
                historical_data = scraper.getStockData()
            except Exception as e:
                raise_custom_exception(ScraperError, message=f"Failed to create StockDataScraper object for {self.stock_symbol}: {e}")

            print(historical_data.columns)
            historical_data = self.get_bollinger_bands(historical_data)
            
            for i in range(1, 20):
                historical_data[f'Close_Lag{i}'] = historical_data['Close'].shift(i)
            
            historical_data.drop(historical_data.head(20).index, inplace=True)
            
            self.plot_stock_data(historical_data)
            self.stock_data = historical_data
            
            historical_data.to_csv(f'./stock_data/{self.stock_symbol}_data.csv')
            self.logger.info(f"Saved stock data to CSV file: {self.stock_symbol}_data.csv")
            
            return historical_data
        
        except CustomException as e:
            self.logger.error(f"A custom error occurred at StockDataAnalayser->fetch_and_plot_stock_data: {e}")
            return None
        except Exception as e:
            self.logger.error(f"An error occurred at StockDataAnalayser->fetch_and_plot_stock_data: {e}")
            return None
    
    def getSeasonalDecomposition(self, stock_data):
        """
        Perform seasonal decomposition on stock data.
        """
        result = seasonal_decompose(stock_data['Close'], model='additive', period=1)
        return result
    
    

    def get_bollinger_bands(self,data, bollinger_window=20):
        """
        Calculate Bollinger Bands for a given stock data.
        """
        rolling_mean = data['Close'].rolling(window=bollinger_window).mean()
        rolling_std = data['Close'].rolling(window=bollinger_window).std()
        data['Bollinger_Upper'] = rolling_mean + (2 * rolling_std)
        data['Bollinger_Lower'] = rolling_mean - (2 * rolling_std)
        return data

    def plot_stock_data(self, data):
        """
        Plot stock data with Bollinger Bands.
        """
        plt.figure(figsize=(12, 6))
        plt.plot(data['Close'], label='Close Price', color='blue')
        plt.plot(data['Bollinger_Upper'], label='Bollinger Upper Band', color='red')
        plt.plot(data['Bollinger_Lower'], label='Bollinger Lower Band', color='green')
        plt.title(f'{self.stock_symbol} Bollinger Bands')
        plt.xlabel('Date')
        plt.ylabel('Price')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

In [4]:
from stock_data_analyser import StockDataAnalyser
analyser=StockDataAnalyser(stock_symbol='MSFT')
analyser.fetch_and_plot_stock_data()

2025-03-31 20:28:20 - StockDataAnalyser - INFO - Fetching stock data for MSFT...
2025-03-31 20:28:20 - StockDataScraper - INFO - Fetching stock data for MSFT...
2025-03-31 20:28:23 - StockDataScraper - INFO - Fetched stock data for MSFT successfully...


Index(['Open', 'High', 'Low', 'Close', 'Volume'], dtype='object')


2025-03-31 20:28:23 - StockDataAnalyser - INFO - Saved stock data to CSV file: MSFT_data.csv


,Open,High,Low,Close,Volume,Bollinger_Upper,Bollinger_Lower,Close_Lag1,Close_Lag2,Close_Lag3,...,Close_Lag11,Close_Lag12,Close_Lag13,Close_Lag14,Close_Lag15,Close_Lag16,Close_Lag17,Close_Lag18,Close_Lag19,Trend
2025-02-28,392.655,397.630,386.570,396.99,32845658,401.112777,377.232223,388.49,388.61,401.02,...,388.70,383.52,387.82,386.84,391.26,393.08,395.16,389.97,390.58,NaN
2025-02-27,401.265,405.740,392.170,392.53,21127406,401.290248,377.249752,396.99,388.49,388.61,...,388.56,388.70,383.52,387.82,386.84,391.26,393.08,395.16,389.97,NaN
2025-02-26,398.010,403.600,394.245,399.73,19618954,402.658175,376.857825,392.53,396.99,388.49,...,378.77,388.56,388.70,383.52,387.82,386.84,391.26,393.08,395.16,NaN
2025-02-25,401.100,401.915,396.700,397.90,29387402,403.091523,376.698477,399.73,392.53,396.99,...,383.27,378.77,388.56,388.70,383.52,387.82,386.84,391.26,393.08,NaN
2025-02-24,408.510,409.370,399.320,404.00,26443656,405.023243,375.858757,397.90,399.73,392.53,...,380.45,383.27,378.77,388.56,388.70,383.52,387.82,386.84,391.26,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-06-25,197.800,200.610,195.470,200.34,27803933,216.881615,196.543385,196.33,198.44,203.51,...,207.07,208.35,208.04,203.92,202.88,211.60,208.75,211.75,202.54,NaN
2020-06-24,201.600,203.250,196.560,197.84,36740647,217.251824,195.703176,200.34,196.33,198.44,...,213.67,207.07,208.35,208.04,203.92,202.88,211.60,208.75,211.75,NaN
2020-06-23,202.090,203.950,201.430,201.91,30917447,216.644129,195.326871,197.84,200.34,196.33,...,214.32,213.67,207.07,208.35,208.04,203.92,202.88,211.60,208.75,NaN
2020-06-22,195.790,200.760,195.230,200.57,32818929,216.414735,194.738265,201.91,197.84,200.34,...,212.83,214.32,213.67,207.07,208.35,208.04,203.92,202.88,211.60,NaN


In [ ]:
analyser.stock_data.info()

In [15]:
import plotly.express as px
import pandas as pd
# Ensure index is in datetime format
data = analyser.stock_data.iloc[:300]
data.index = pd.to_datetime(data.index)

# Create interactive line chart
fig = px.line(data, x=data.index, y='Close', title="Stock Price Trends")
# fig= pd.line(data, x=data.index,)
# Improve X and Y axes
fig.update_xaxes(title="Date", rangeslider_visible=True)
fig.update_yaxes(title="Price")

# Show interactive chart
fig.show()


In [11]:
import numpy as np
def calculate_ema_macd_adx(df, short_window=12, long_window=26, signal_window=9, adx_window=14):
    """
    Calculates EMA, MACD, and ADX for the given DataFrame.
    :param df: Pandas DataFrame containing a 'Close' column.
    :param short_window: Short EMA window (default: 12)
    :param long_window: Long EMA window (default: 26)
    :param signal_window: Signal line window for MACD (default: 9)
    :param adx_window: ADX smoothing period (default: 14)
    :return: DataFrame with added EMA, MACD, and ADX columns.
    """
    df = df.copy()
    
    # Calculate EMA
    df['EMA'] = df['Close'].ewm(span=short_window, adjust=False).mean()
    
    # Calculate MACD
    df['EMA_Short'] = df['Close'].ewm(span=short_window, adjust=False).mean()
    df['EMA_Long'] = df['Close'].ewm(span=long_window, adjust=False).mean()
    df['MACD'] = df['EMA_Short'] - df['EMA_Long']
    df['MACD_Signal'] = df['MACD'].ewm(span=signal_window, adjust=False).mean()
    
    # Calculate ADX
    df['High-Low'] = df['High'] - df['Low']
    df['High-Close'] = np.abs(df['High'] - df['Close'].shift(1))
    df['Low-Close'] = np.abs(df['Low'] - df['Close'].shift(1))
    df['TR'] = df[['High-Low', 'High-Close', 'Low-Close']].max(axis=1)
    df['ATR'] = df['TR'].rolling(window=adx_window).mean()
    
    df['+DM'] = np.where(df['High'].diff() > df['Low'].diff(), df['High'].diff(), 0)
    df['-DM'] = np.where(df['Low'].diff() > df['High'].diff(), df['Low'].diff(), 0)
    df['+DI'] = (100 * df['+DM'].rolling(window=adx_window).mean()) / df['ATR']
    df['-DI'] = (100 * df['-DM'].rolling(window=adx_window).mean()) / df['ATR']
    df['DX'] = (np.abs(df['+DI'] - df['-DI']) / (df['+DI'] + df['-DI'])) * 100
    df['ADX'] = df['DX'].rolling(window=adx_window).mean()
    
    return df


In [37]:
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import seasonal_decompose
data=calculate_ema_macd_adx(data, short_window=15, long_window=30)
sd1=seasonal_decompose(data['Close'], model='multiplicative', period=30)
sd2=seasonal_decompose(data['Close'], model='additive', period=30)
# Create figure
fig = go.Figure()

# Add Close Price line
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Close Price', line=dict(color='blue')))

# Add Bollinger Upper Band
fig.add_trace(go.Scatter(x=data.index, y=data['Bollinger_Upper'], mode='lines', name='Bollinger Upper Band', line=dict(color='red')))

# Add Bollinger Lower Band
fig.add_trace(go.Scatter(x=data.index, y=data['Bollinger_Lower'], mode='lines', name='Bollinger Lower Band', line=dict(color='green')))

# Add seasonal decomposition components
# fig.add_trace(go.Scatter(x=data.index, y=data['EMA'], mode='lines', name='EMA', line=dict(color='orange')))

# fig.add_trace(go.Scatter(x=data.index, y=sd2.trend.dropna(), mode='lines', name='Trend', line=dict(color='yellow')))
fig.add_trace(go.Scatter(x=data.index, y=data['EMA_Long'], mode='lines', name='EMA_long', line=dict(color='yellow')))

fig.add_trace(go.Scatter(x=data.index, y=data['EMA_Short'], mode='lines', name='EMA-short', line=dict(color='orange')))

# Customize layout
fig.update_layout(
    title="Stock Price with Bollinger Bands",
    xaxis_title="Date",
    yaxis_title="Price",
    legend=dict(x=0, y=1),
    # xaxis=dict(rangeslider=dict(visible=True))  # Enables zooming and panning
)

# Show the figure
fig.show()


In [26]:
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import seasonal_decompose

# Assuming `data` is the DataFrame containing Close, EMA, MACD, Signal, and ADX
data = calculate_ema_macd_adx(data)

# Seasonal decomposition
sd1 = seasonal_decompose(data['Close'], model='multiplicative', period=30)
sd2 = seasonal_decompose(data['Close'], model='additive', period=30)

# Create figure
fig = go.Figure()

# Add Close Price line
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Close Price', line=dict(color='blue')))

# Add EMA
fig.add_trace(go.Scatter(x=data.index, y=data['EMA'], mode='lines', name='EMA (12)', line=dict(color='red', dash='dash')))

# Add MACD
# fig.add_trace(go.Scatter(x=data.index, y=data['MACD'], mode='lines', name='MACD', line=dict(color='green')))

# Add MACD Signal Line
# fig.add_trace(go.Scatter(x=data.index, y=data['MACD_Signal'], mode='lines', name='MACD Signal Line', line=dict(color='orange', dash='dash')))

# Add ADX
# fig.add_trace(go.Scatter(x=data.index, y=data['ADX'], mode='lines', name='ADX', line=dict(color='purple')))

# Add Seasonal Decomposition (Multiplicative Trend)
fig.add_trace(go.Scatter(x=data.index, y=sd1.trend.dropna(), mode='lines', name='Multiplicative Trend', line=dict(color='orange')))

# Add Seasonal Decomposition (Additive Trend)
fig.add_trace(go.Scatter(x=data.index, y=sd2.trend.dropna(), mode='lines', name='Additive Trend', line=dict(color='yellow')))

# Customize layout
fig.update_layout(
    title="Stock Price with EMA, MACD, ADX, and Seasonal Decomposition",
    xaxis_title="Date",
    yaxis_title="Price",
    legend=dict(x=0, y=1),
    xaxis=dict(rangeslider=dict(visible=True))  # Enables zooming and panning
)

# Show the figure
fig.show()


In [9]:
df.index = pd.to_datetime(df.index)
# Convert the columns to numeric
df = df.apply(pd.to_numeric) 
df.head()

,Open,High,Low,Close,Volume
2025-03-28,388.080,389.13,376.930,378.80,21632016
2025-03-27,390.130,392.24,387.395,390.58,13766761
2025-03-26,395.000,395.31,388.570,389.97,16132906
2025-03-25,393.915,396.36,392.640,395.16,15774968
2025-03-24,395.400,395.40,389.810,393.08,21004548


In [ ]:
df = pd.DataFrame.from_dict(scraper.stock_data['Time Series (Daily)'], orient='index')

df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']

csv_path = "stock_data.csv"
df.to_csv(csv_path)

print(f"CSV file saved at: {csv_path}")

CSV file saved at: stock_data.csv


In [4]:
data

{'Symbol': 'MSFT',
 'AssetType': 'Common Stock',
 'Name': 'Microsoft Corporation',
 'Description': "Microsoft Corporation is an American multinational technology company which produces computer software, consumer electronics, personal computers, and related services. Its best known software products are the Microsoft Windows line of operating systems, the Microsoft Office suite, and the Internet Explorer and Edge web browsers. Its flagship hardware products are the Xbox video game consoles and the Microsoft Surface lineup of touchscreen personal computers. Microsoft ranked No. 21 in the 2020 Fortune 500 rankings of the largest United States corporations by total revenue; it was the world's largest software maker by revenue as of 2016. It is considered one of the Big Five companies in the U.S. information technology industry, along with Google, Apple, Amazon, and Facebook.",
 'CIK': '789019',
 'Exchange': 'NASDAQ',
 'Currency': 'USD',
 'Country': 'USA',
 'Sector': 'TECHNOLOGY',
 'Indust

In [7]:
from stock_data_analyser import StockDataAnalyser
analyser=StockDataAnalyser(stock_symbol='AAPL')
analyser.fetch_and_plot_stock_data()

2025-04-02 19:58:33 - StockDataAnalyser - INFO - Fetching stock data for AAPL...
2025-04-02 19:58:33 - StockDataScraper - INFO - Fetching stock data for AAPL...
2025-04-02 19:58:37 - StockDataScraper - INFO - Fetched stock data for AAPL successfully...


,Open,High,Low,Close,Volume,EMA,Bollinger_Upper,Bollinger_Lower,Close_Lag1,Close_Lag2,...,Close_Lag12,Close_Lag13,Close_Lag14,Close_Lag15,Close_Lag16,Close_Lag17,Close_Lag18,Close_Lag19,Trend,Stock_name
2025-03-04,237.705,240.0700,234.680,235.93,53798062,224.784136,239.290079,204.582921,235.74,235.33,...,214.10,218.27,220.73,223.75,221.53,223.85,217.90,222.13,NaN,AAPL
2025-03-03,241.790,244.0272,236.112,238.03,47183985,226.045647,241.519904,203.943096,235.93,235.74,...,215.24,214.10,218.27,220.73,223.75,221.53,223.85,217.90,NaN,AAPL
2025-02-28,236.950,242.0900,230.200,241.84,56833360,227.549871,244.396215,203.460785,238.03,235.93,...,212.69,215.24,214.10,218.27,220.73,223.75,221.53,223.85,NaN,AAPL
2025-02-27,239.410,242.4600,237.060,237.30,41153639,228.478455,245.923835,203.278165,241.84,238.03,...,214.00,212.69,215.24,214.10,218.27,220.73,223.75,221.53,NaN,AAPL
2025-02-26,244.330,244.9800,239.130,240.36,44433564,229.610030,247.930641,203.154359,237.30,241.84,...,213.49,214.00,212.69,215.24,214.10,218.27,220.73,223.75,NaN,AAPL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-06-29,353.250,362.1736,351.280,361.78,32661519,378.613988,398.655260,358.688740,364.80,364.11,...,386.09,385.31,393.43,388.00,389.09,371.38,370.46,379.24,NaN,AAPL
2020-06-26,364.410,365.3200,353.020,353.63,51314211,376.234561,400.290851,354.492149,361.78,364.80,...,390.90,386.09,385.31,393.43,388.00,389.09,371.38,370.46,NaN,AAPL
2020-06-25,360.700,365.0000,357.570,364.84,34380628,375.149365,400.500653,353.720347,353.63,361.78,...,388.23,390.90,386.09,385.31,393.43,388.00,389.09,371.38,NaN,AAPL
2020-06-24,365.000,368.7900,358.520,360.06,48155849,373.712282,401.040237,352.048763,364.84,353.63,...,381.91,388.23,390.90,386.09,385.31,393.43,388.00,389.09,NaN,AAPL


In [1]:
from stock_data_analyser import StockDataAnalyser
analyser=StockDataAnalyser(stock_symbol='AAPL')
analyser.fetch_from_db_and_analyze()

2025-04-02 20:01:58 - StockDataAnalyser - ERROR - An error occurred at StockDataAnalayser->_fetch_from_sql: (psycopg2.errors.UndefinedTable) relation "aapl" does not exist
LINE 1: SELECT * FROM aapl
                      ^

[SQL: SELECT * FROM aapl]
(Background on this error at: https://sqlalche.me/e/20/f405)
2025-04-02 20:01:58 - StockDataAnalyser - ERROR - An error occurred at StockDataAnalayser->fetch_from_sql_and_analyze: 'NoneType' object has no attribute 'index'


SELECT * FROM aapl


In [5]:
from sqlalchemy import create_engine
import pandas as pd
import os
import sys
# load env
from dotenv import load_dotenv
load_dotenv()
# load env
from data_scraper import StockDataScraper
import pandas as pd
# Define your database credentials
db_url = os.getenv('POSTGRES_URI')

# Create SQLAlchemy engine
engine = create_engine(db_url)


In [6]:
analyser.stock_data.to_sql("stock_data",engine, if_exists="replace", index=True)

180

In [16]:
# Read back from SQL to confirm
df = pd.read_sql("SELECT * FROM stock_data", engine)
df.index=df['index']
print(df.iloc[0:1])


                 index    Open    High    Low   Close    Volume         EMA  \
index                                                                         
2025-02-28  2025-02-28  236.95  242.09  230.2  241.84  56833360  226.915551   

            Bollinger_Upper  Bollinger_Lower  Close_Lag1  ...  Close_Lag12  \
index                                                     ...                
2025-02-28       244.396215       203.460785      238.03  ...       212.69   

            Close_Lag13  Close_Lag14  Close_Lag15  Close_Lag16  Close_Lag17  \
index                                                                         
2025-02-28       215.24        214.1       218.27       220.73       223.75   

            Close_Lag18  Close_Lag19  Trend  Stock_name  
index                                                    
2025-02-28       221.53       223.85    NaN        AAPL  

[1 rows x 30 columns]
